**Import Required Libraries**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

**Load Project Utilities & Logging**

In [0]:
%run /Workspace/airline_etl_pipeline/2_utilities
%run /Workspace/airline_etl_pipeline/3_pipeline_log

In [0]:
print(catalog, bronze_schema, silver_schema, gold_schema)
print("Landing:", landing_path)

**Define table names**

In [0]:
bronze_table   = f"{catalog}.{bronze_schema}.flights_raw"
staging_bronze = f"{catalog}.{bronze_schema}.staging_flights"
silver_fact    = f"{catalog}.{silver_schema}.fact_flights"
staging_silver = f"{catalog}.{silver_schema}.staging_flights"
silver_carrier = f"{catalog}.{silver_schema}.dim_carrier"
silver_airport = f"{catalog}.{silver_schema}.dim_airport"
gold_fact      = f"{catalog}.{gold_schema}.fact_flight_delays"
gold_agg       = f"{catalog}.{gold_schema}.agg_delay_summary"

print("Staging Bronze :", staging_bronze)
print("Staging Silver :", staging_silver)

**Initialize logging**

In [0]:
init_log_table()
run_id     = str(uuid.uuid4())
start_time = datetime.utcnow()

## Bronze

Read today's new CSV files from S3 landing zone.

In [0]:
# Read new daily files from S3 landing zone
df = (
    spark.read
    .options(header=True, inferSchema=True)
    .csv(f"{landing_path}*/*.csv")
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

print("New rows today:", df.count())
df.show(5)

In [0]:
# Append to main Bronze table (preserves full history)
df.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("append") \
    .saveAsTable(bronze_table)

print(f"✅ Appended to Bronze → {bronze_table}")

### Staging table — only today's rows (same pattern as FMCG project)

In [0]:
df.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(staging_bronze)

print(f"✅ Staging Bronze → {staging_bronze}")

### Move files from landing/ → processed/

In [0]:
date_folders = dbutils.fs.ls(landing_path)
moved = 0
for folder in date_folders:
    try:
        files = dbutils.fs.ls(folder.path)
        for file_info in files:
            dest = file_info.path.replace("landing/flights/", "processed/flights/")
            dbutils.fs.mv(file_info.path, dest, True)
            moved += 1
    except:
        pass

print(f"✅ Moved {moved} files → processed/")

## Silver

Read from staging only — not full Bronze.

In [0]:
df_flights = spark.sql(f"SELECT * FROM {staging_bronze}")
print("Staging rows:", df_flights.count())
df_flights.show(2)

**Transformations**

In [0]:
# 1. Drop nulls on key columns
df_flights = df_flights.filter(
    F.col("FlightDate").isNotNull() &
    F.col("Origin").isNotNull() &
    F.col("Dest").isNotNull()
)

# 2. Cast delay columns
df_flights = df_flights \
    .withColumn("DepDelay",
        F.when(F.col("DepDelay").cast("double") < -120, F.lit(None))
         .otherwise(F.col("DepDelay").cast("double"))) \
    .withColumn("ArrDelay", F.col("ArrDelay").cast("double"))

# 3. Fill delay cause nulls with 0.0
for col_name in ["CarrierDelay", "WeatherDelay", "NASDelay",
                 "SecurityDelay", "LateAircraftDelay"]:
    df_flights = df_flights.withColumn(
        col_name,
        F.coalesce(F.col(col_name).cast("double"), F.lit(0.0))
    )

# 4. CancellationCode
df_flights = df_flights.withColumn(
    "CancellationCode",
    F.when(F.col("Cancelled") == 1, F.col("CancellationCode"))
     .otherwise(F.lit("N"))
)

# 5. Parse FlightDate
df_flights = df_flights.withColumn(
    "FlightDate", F.to_date("FlightDate", "yyyy-MM-dd")
)

# 6. Delay bucket
df_flights = df_flights.withColumn(
    "delay_bucket",
    F.when(F.col("ArrDelay") <= 0,   F.lit("on_time"))
     .when(F.col("ArrDelay") <= 15,  F.lit("minor"))
     .when(F.col("ArrDelay") <= 60,  F.lit("moderate"))
     .otherwise(F.lit("severe"))
)

# 7. Primary delay cause
df_flights = df_flights.withColumn(
    "primary_delay_cause",
    F.when(F.col("WeatherDelay") > 0,      F.lit("weather"))
     .when(F.col("CarrierDelay") > 0,      F.lit("carrier"))
     .when(F.col("NASDelay") > 0,          F.lit("nas"))
     .when(F.col("LateAircraftDelay") > 0, F.lit("late_aircraft"))
     .otherwise(F.lit("none"))
)

# 8. SHA surrogate key
df_flights = df_flights.withColumn(
    "flight_key",
    F.sha2(
        F.concat_ws("_",
            F.col("FlightDate"),
            F.col("Reporting_Airline"),
            F.col("Flight_Number_Reporting_Airline"),
            F.col("Origin"),
            F.col("Dest")
        ), 256
    )
)

# 9. Drop duplicates
df_flights = df_flights.dropDuplicates(["flight_key"])

print("Clean incremental rows:", df_flights.count())

**Update dim_carrier and dim_airport with any new entries**

In [0]:
# Upsert dim_carrier
new_carriers = df_flights.select(
    F.col("Reporting_Airline").alias("carrier_code")
).distinct()
DeltaTable.forName(spark, silver_carrier).alias("t") \
    .merge(new_carriers.alias("s"), "t.carrier_code = s.carrier_code") \
    .whenNotMatchedInsertAll().execute()
print("✅ dim_carrier updated")

# Upsert dim_airport
new_airports = df_flights.select(
    F.col("Origin").alias("airport_code"),
    F.col("OriginCityName").alias("city_name"),
    F.col("OriginState").alias("state")
).union(
    df_flights.select(
        F.col("Dest").alias("airport_code"),
        F.col("DestCityName").alias("city_name"),
        F.col("DestState").alias("state")
    )
).distinct()
DeltaTable.forName(spark, silver_airport).alias("t") \
    .merge(new_airports.alias("s"), "t.airport_code = s.airport_code") \
    .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
print("✅ dim_airport updated")

**Upsert Silver Fact + write Silver staging**

In [0]:
# Upsert Silver Fact
DeltaTable.forName(spark, silver_fact).alias("silver") \
    .merge(
        df_flights.alias("staging"),
        "silver.flight_key = staging.flight_key"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()
print(f"✅ Silver upserted → {silver_fact}")

# Write Silver staging (incremental only — used by Gold step)
df_flights.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(staging_silver)
print(f"✅ Silver staging → {staging_silver}")

## Gold

### Incremental recalculation — same pattern as your FMCG project

In [0]:
# Find which dates arrived in this batch
df_staging = spark.sql(f"SELECT FlightDate FROM {staging_silver}")

incremental_dates = df_staging.select(
    F.col("FlightDate").alias("arrival_date")
).distinct()

print("Dates in this incremental batch:")
incremental_dates.orderBy("arrival_date").show()

incremental_dates.createOrReplaceTempView("incremental_dates")

In [0]:
# Pull only affected dates from Silver for recalculation
df_affected = spark.sql(f"""
    SELECT sf.*
    FROM {silver_fact} sf
    INNER JOIN incremental_dates id
        ON sf.FlightDate = id.arrival_date
""")

print("Affected rows:", df_affected.count())

In [0]:
# Window function — recalculate latest pattern for affected routes
w = Window.partitionBy("Origin", "Dest", "Reporting_Airline") \
          .orderBy(F.col("FlightDate").desc())

df_gold_inc = df_affected \
    .withColumn("rn", F.row_number().over(w)) \
    .withColumn("is_latest_record", (F.col("rn") == 1).cast("int")) \
    .drop("rn")

# Upsert Gold Fact
DeltaTable.forName(spark, gold_fact).alias("gold") \
    .merge(
        df_gold_inc.alias("new"),
        "gold.flight_key = new.flight_key"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()
print(f"✅ Gold fact upserted → {gold_fact}")

**Recalculate aggregation for affected dates only**

In [0]:
df_agg_inc = df_gold_inc.groupBy(
    "FlightDate", "Reporting_Airline",
    "Origin", "Dest",
    "delay_bucket", "primary_delay_cause"
).agg(
    F.count("*").alias("total_flights"),
    F.avg("ArrDelay").alias("avg_arr_delay"),
    F.avg("DepDelay").alias("avg_dep_delay"),
    F.max("ArrDelay").alias("max_arr_delay"),
    F.sum(F.when(F.col("Cancelled") == 1, 1).otherwise(0)).alias("cancellations"),
    F.sum(F.when(F.col("Diverted") == 1, 1).otherwise(0)).alias("diversions")
)

DeltaTable.forName(spark, gold_agg).alias("agg") \
    .merge(
        df_agg_inc.alias("new"),
        """agg.FlightDate        = new.FlightDate
           AND agg.Reporting_Airline = new.Reporting_Airline
           AND agg.Origin            = new.Origin
           AND agg.Dest              = new.Dest
           AND agg.delay_bucket      = new.delay_bucket"""
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()
print(f"✅ Gold agg upserted → {gold_agg}")

**Write pipeline log**

In [0]:
write_log(
    job_name     = "incremental_load_fact",
    layer        = "bronze→silver→gold",
    run_id       = run_id,
    start_time   = start_time,
    rows_read    = df_flights.count(),
    rows_written = df_gold_inc.count(),
    status       = "success"
)

## Cleanup staging tables

In [0]:
%sql
DROP TABLE IF EXISTS airline.bronze.staging_flights;
DROP TABLE IF EXISTS airline.silver.staging_flights;

**Verify row counts**

In [0]:
%sql
SELECT 'bronze'      AS layer, COUNT(*) AS rows FROM airline.bronze.flights_raw
UNION ALL
SELECT 'silver_fact' AS layer, COUNT(*) AS rows FROM airline.silver.fact_flights
UNION ALL
SELECT 'gold_fact'   AS layer, COUNT(*) AS rows FROM airline.gold.fact_flight_delays
UNION ALL
SELECT 'gold_agg'    AS layer, COUNT(*) AS rows FROM airline.gold.agg_delay_summary
UNION ALL
SELECT 'logs'        AS layer, COUNT(*) AS rows FROM airline.gold.pipeline_logs;